# One supercell, every backend

This notebook runs CM1's bundled `supercell_kessler` case (40×40×40 at 1 km, Kessler
warm rain, TKE turbulence, open radiative boundaries, a moving domain) for 30 minutes
on every implementation in this repository, through one driver (`cm1drive`):

| backend | what it is |
|---|---|
| `fortran` | CM1 itself in double precision: state dumps written by `cxx/scripts/make_refs.sh` |
| `cm1k` | the C++/Kokkos port (`cxx/`) through its Python module |
| `cm1x:numpy`, `cm1x:jax:jit`, `cm1x:torch` | the array-API port (`ports/array`) on each array library (jax compiled with `jax.jit`, torch eager) |

Each port initializes the case from the namelist itself, so the comparison covers the
initial sounding, the bubble and every time step. See `ports/driver/README.md` for
how to install the pieces.

**Settings** (environment variables): `CM1_REF_DIR` the Fortran dumps of the case
(default `refs/supercell_kessler`), `CM1_STEPS` (default 300 = 30 min),
`CM1_EVERY` the snapshot interval in steps (default 30, as the dumps), and
`CM1_BACKENDS` a comma-separated list (default: all of the above that import here;
`cm1x:jax` eager, ~3 s/step, and `cm1x:torch:jit`, minutes of compiling, are opt-in).

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display


def find_root():
    p = Path.cwd().resolve()
    for q in (p, *p.parents):
        if (q / "ports" / "driver" / "cm1drive.py").is_file():
            return q
    raise RuntimeError("run this notebook inside the CM1 repository")


ROOT = find_root()
sys.path.insert(0, str(ROOT / "ports" / "driver"))
import cm1drive

CASE = "supercell_kessler"
REF_DIR = Path(os.environ.get("CM1_REF_DIR", ROOT / "refs" / CASE))
STEPS = int(os.environ.get("CM1_STEPS", "300"))
EVERY = int(os.environ.get("CM1_EVERY", "30"))
DEFAULT = ["fortran", "cm1k", "cm1x:numpy", "cm1x:jax:jit", "cm1x:torch"]
have = cm1drive.available(REF_DIR)
if os.environ.get("CM1_BACKENDS"):
    BACKENDS = os.environ["CM1_BACKENDS"].split(",")
else:
    BACKENDS = [b for b in DEFAULT if b in have]
missing = [b for b in DEFAULT if b not in have]
print("running:", ", ".join(BACKENDS))
if missing:
    print("not available here:", ", ".join(missing))
assert "fortran" in BACKENDS, (
    f"no Fortran dumps in {REF_DIR} (cxx/scripts/make_refs.sh)"
)

## Run

The Fortran "run" reads the dumps; the others step the model.

In [ ]:
runs = {}
for b in BACKENDS:
    r = cm1drive.run(CASE, b, STEPS, every=EVERY, ref_dir=REF_DIR)
    runs[b] = r
    print(f"{r.name:14s} init {r.init_seconds:6.1f} s, {r.seconds_per_step:.3f} s/step")
ref = runs["fortran"]
grid = ref.grid

## The storm

After 30 minutes the bubble has grown into a deep convective cell. Below, from CM1
itself: the largest vertical velocity in each column, rain water at the lowest level,
and an east-west cross-section of cloud and rain through the strongest updraft.

In [ ]:
last = ref.steps[-1]
minutes = last * grid.dtl / 60
w = ref["wa"]  # [k, j, i] on w levels (nz + 1)
qc, qr = ref["qa_qc"] * 1e3, ref["qa_qr"] * 1e3  # g/kg
k, j, i = np.unravel_index(np.argmax(w), w.shape)
zf = np.arange(grid.nz + 1) * grid.dz / 1000

fig, ax = plt.subplots(1, 3, figsize=(15, 4.2), constrained_layout=True)
pc = ax[0].pcolormesh(grid.x, grid.y, w.max(axis=0), cmap="RdBu_r", vmin=-30, vmax=30)
fig.colorbar(pc, ax=ax[0], label="m/s")
ax[0].set_title("column-max w")
pc = ax[1].pcolormesh(grid.x, grid.y, qr[0], cmap="Blues", vmin=0)
fig.colorbar(pc, ax=ax[1], label="g/kg")
ax[1].set_title("rain water, lowest level")
for a in ax[:2]:
    a.set_aspect("equal")
    a.set_xlabel("x (km)")
    a.set_ylabel("y (km)")
    a.plot(grid.x[i], grid.y[j], "k+", ms=12)
pc = ax[2].pcolormesh(grid.x, grid.z, (qc + qr)[:, j, :], cmap="Greys", vmin=0)
fig.colorbar(pc, ax=ax[2], label="g/kg")
ax[2].contour(
    grid.x, zf, w[:, j, :], levels=[5, 10, 20, 30], colors="r", linewidths=0.8
)
ax[2].set_title(f"cloud + rain (shaded), w (red, 5-30 m/s) at y = {grid.y[j]:.1f} km")
ax[2].set_xlabel("x (km)")
ax[2].set_ylabel("z (km)")
fig.suptitle(f"CM1 (Fortran), supercell_kessler at {minutes:.0f} min")
plt.show()

## Every backend against Fortran

The relative difference of a field is `max|a - b| / max(max|a|, max|b|)` over the
interior points (as in `ports/crosscheck`). Bitwise-identical runs are drawn at the
bottom of the axis.

In [ ]:
display(Markdown(cm1drive.table(runs.values(), ref)))

FLOOR = 1e-17
others = [r for r in runs.values() if r is not ref]
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2), constrained_layout=True)
for r, marker in zip(others, "osD^vP"):
    wst = cm1drive.worst(cm1drive.compare(r, ref))
    t = [s * grid.dtl / 60 for s in wst]
    v = [max(x, FLOOR) for x, _ in wst.values()]
    same = max(x for x, _ in wst.values()) == 0
    ax[0].semilogy(
        t,
        v,
        marker + "-",
        mfc="none",
        ms=9,
        label=r.name + (" (bitwise)" if same else ""),
    )
for tol, label in (
    (cm1drive.LOOSE, "jax / torch tolerance"),
    (cm1drive.BITWISE, "bitwise tolerance"),
):
    ax[0].axhline(tol, color="k", lw=0.6, ls="--")
    ax[0].text(0, tol * 1.5, f"{label} {tol:.0e}", fontsize=8)
ax[0].set_ylim(FLOOR / 3, 1)
ax[0].set_xlabel("minutes")
ax[0].set_ylabel("largest relative difference (any field)")
ax[0].set_title("difference from Fortran CM1")
ax[0].legend()
names = [r.name for r in others]
ax[1].barh(names, [r.seconds_per_step for r in others])
ax[1].set_xlabel("seconds per step (wall)")
ax[1].set_title(f"time stepping, {os.cpu_count()} CPUs")
ax[1].invert_yaxis()
plt.show()

Per field, at the last snapshot (rows: backends, columns: fields; blank where
identical):

In [ ]:
fields = [f for f in ref.fields if f in ref.snapshots[last]]
hdr = "| backend | " + " | ".join(f"`{f}`" for f in fields) + " |"
rows = [hdr, "|---" * (len(fields) + 1) + "|"]
for r in others:
    d = cm1drive.compare(r, ref)
    cells = [f"{d[f][last]:.0e}" if d.get(f, {}).get(last) else "" for f in fields]
    rows.append(f"| {r.name} | " + " | ".join(cells) + " |")
display(Markdown("\n".join(rows)))

## Where they differ

Column-max of `|w - w_fortran|` at the last snapshot, per backend (a blank panel is
bitwise identical). Last-bit differences (from a different `exp`/`log` or summation
order) spread through the domain with the sound waves, and the open radiative
boundaries amplify them; they stay far below the storm's own signal.

In [ ]:
n = len(others)
fig, axs = plt.subplots(
    1, n, figsize=(3.6 * n, 3.4), constrained_layout=True, squeeze=False
)
for a, r in zip(axs[0], others):
    d = np.abs(r["wa"] - w).max(axis=0)
    if d.max() == 0:
        a.text(
            0.5,
            0.5,
            "bitwise\nidentical",
            ha="center",
            va="center",
            transform=a.transAxes,
        )
        a.pcolormesh(grid.x, grid.y, np.full_like(d, np.nan))
    else:
        pc = a.pcolormesh(
            grid.x,
            grid.y,
            np.log10(np.maximum(d, 1e-20)),
            cmap="magma",
            vmin=-16,
            vmax=-4,
        )
        fig.colorbar(pc, ax=a, label="log10 m/s")
    a.set_title(r.name)
    a.set_aspect("equal")
    a.set_xlabel("x (km)")
axs[0][0].set_ylabel("y (km)")
fig.suptitle(f"column-max |w - w_fortran| at {minutes:.0f} min")
plt.show()

## Check

The last cell fails (and so does CI) when an implementation leaves its expected
agreement with Fortran (`cm1drive.tolerance`): bitwise (1e-12) for cm1k on the CPU,
and for cm1x on numpy when numpy calls glibc's `exp`/`log`/`pow` like CM1 (numpy's
AVX-512 versions off, as in CI). jax and torch use their own vectorized math, and
this case's open boundaries amplify last-bit differences, so they are held to 1e-5,
still far below anything visible in the storm.

In [ ]:
report = os.environ.get("CM1_REPORT")
if report:  # CI puts this in the job summary
    Path(report).write_text(
        f"### supercell_kessler, {minutes:.0f} min, every backend vs Fortran\n\n"
        + cm1drive.table(runs.values(), ref)
        + "\n"
    )
bad = [m for m in (cm1drive.check(r, ref) for r in others) if m]
assert not bad, "\n".join(bad)
print("all backends agree with Fortran CM1 within their tolerance")